In [1]:
import os
os.getcwd()
os.chdir("c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\ImageClass_MLOps_DVCpipeline\\deeplearning_MLOps_DVC")

In [2]:
%pwd

'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\ImageClass_MLOps_DVCpipeline\\deeplearning_MLOps_DVC'

In [3]:
# os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path


In [5]:
## preparing configuration manager

# from CNNclassifier.entity.config_entity import TrainingConfig
from CNNclassifier.constants import *
from CNNclassifier.utils.common import read_yaml, create_directories

class ConfigurationMananger:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH, 
                 params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks

        model_chkpt_dir = os.path.dirname(config.checkpoint_model_filepath)

        create_directories(
            [Path(config.tensorboard_root_log_dir),
             Path(model_chkpt_dir)]
             )
        
        prepare_callback_config = PrepareCallbacksConfig(
            root_dir= Path(config.root_dir),
            tensorboard_root_log_dir= Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath= Path(config.checkpoint_model_filepath)
        )
    
        return prepare_callback_config
    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config


In [6]:
from zipfile import ZipFile
import os
import urllib.request as request
import tensorflow as tf
import time
from CNNclassifier import logger

In [7]:
tf.config.run_functions_eagerly(True)

In [8]:
class PrepareCallback:
    def __init__(self, config = PrepareCallbacksConfig):
        self.config = config

    @property
    def _create_tb_callback(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}"
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    
    @property
    def _create_chkpt_callback(self):
        
        return(tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        ))
    
    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callback, 
            self._create_chkpt_callback]
    

In [ ]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        print(f"Base model loaded from: {self.config.updated_base_model_path}")
    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)


    def train(self):#, callback_list: list
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=1,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
            # callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

        training_data = self.config.training_data
        image_size = self.config.params_image_size[:-1]
        batch_size = self.config.params_batch_size

        logger.info(f"Training completed for data: {training_data} with image size: {image_size} and batch size: {batch_size}")

In [10]:
try:
    config = ConfigurationMananger()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        # callback_list=callback_list
    )
    
except Exception as e:
    raise e



2026-06-02 16:24:40,827 - CNNclassifier_logger - INFO - CNNclassifier_logger - common.py - read_yaml - 44 - yaml file: config\config.yaml loaded successfully
2026-06-02 16:24:40,829 - CNNclassifier_logger - INFO - CNNclassifier_logger - common.py - read_yaml - 44 - yaml file: params.yaml loaded successfully
2026-06-02 16:24:40,830 - CNNclassifier_logger - INFO - CNNclassifier_logger - common.py - create_directories - 202 - created directory at: artifacts\prepare_callbacks\tensorboard_log_dir
2026-06-02 16:24:40,831 - CNNclassifier_logger - INFO - CNNclassifier_logger - common.py - create_directories - 202 - created directory at: artifacts\prepare_callbacks\checkpoint_dir
2026-06-02 16:24:40,832 - CNNclassifier_logger - INFO - CNNclassifier_logger - common.py - create_directories - 202 - created directory at: artifacts\training
2026-06-02 16:24:41,028 - tensorflow - WARNING - tensorflow - config.py - list_physical_devices - 464 - TensorFlow GPU support is not available on native Windows

TypeError: Training.train() missing 1 required positional argument: 'callback_list'

In [ ]:

print(tf.executing_eagerly())

True


In [ ]:
tf.data.experimental.enable_debug_mode()

In [ ]:
%pwd
